<a href="https://colab.research.google.com/github/ameliewei/NomNom/blob/main/NomNom.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gradio huggingface_hub sentence-transformers torch -q


In [ ]:
import gradio as gr
from huggingface_hub import InferenceClient
from sentence_transformers import SentenceTransformer
import torch
import re #so we can call it later for buttons - builds expressions
import os #for google maps api
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["GOOGLE_MAPS_API_KEY"] = userdata.get("GOOGLE_MAPS_API_KEY")
GOOGLE_MAPS_API_KEY = os.environ.get("GOOGLE_MAPS_API_KEY")

# This is the same pattern from the Generative AI lesson! It uses the
# Inference Provider API to send your messages to an AI model and get
# a response back. Swap out the model below for a different one if
# you want to experiment!
#
# Note: if this Space doesn't already have one, you'll need to add an
# HF_TOKEN secret in the Space's Settings tab for this to work
# (Settings -> Variables and secrets -> New secret).



client = InferenceClient("Qwen/Qwen2.5-7B-Instruct", bill_to="kode-with-klossy")



with open("knowledge.txt", "r", encoding="utf-8") as file:
 knowledge_text = file.read()



def preprocess_text(text):
 cleaned_text = text.strip()
 chunks = cleaned_text.split("\n")
 cleaned_chunks = []
 for chunk in chunks:
   stripped_chunk = chunk.strip()
   if len(stripped_chunk) > 0:
     cleaned_chunks.append(stripped_chunk)
 return cleaned_chunks



cleaned_chunks = preprocess_text(knowledge_text)
model = SentenceTransformer('all-MiniLM-L6-v2')



def create_embeddings(text_chunks):
 chunk_embeddings = model.encode(text_chunks, convert_to_tensor=True)
 return chunk_embeddings



chunk_embeddings = create_embeddings(cleaned_chunks) # Complete this line



def get_top_chunks(query, chunk_embeddings, text_chunks):
 query_embedding = model.encode(query, convert_to_tensor=True) # Complete this line
 query_embedding_normalized = query_embedding / query_embedding.norm()
 chunk_embeddings_normalized = chunk_embeddings / chunk_embeddings.norm(dim=1, keepdim=True)
 similarities = torch.matmul(chunk_embeddings_normalized, query_embedding_normalized) # Complete this line
 top_indices = torch.topk(similarities, k=4).indices
 top_chunks = []
 for i in top_indices:
   chunk = text_chunks[i]
   top_chunks.append(chunk)



 return top_chunks



def extract_known_restaurant_names(reply, chunks, max_names=3): #restaurant names from our knowledge base to make sure buttons work
   names = [m.group(1) for c in chunks if (m := re.match(r"\[(?:CHAIN|SMALL BUSINESS)\]\s*(.+?)\s*\|", c)) and m.group(1) in reply] #does it start with chain/small business?
   return list(dict.fromkeys(names))[:max_names] #if so, saved as m --> if there's a name and mentioned, read to go on buttons



def respond(message, history):
   top_chunks = get_top_chunks(message, chunk_embeddings, cleaned_chunks)
   context = "\n\n".join(top_chunks)



   system_prompt = f"""You are a personalized restaurant finder and recommendation assistant. Before answering any restaurant-related request, first ask the user to choose a location from exactly these three options: LOS ANGELES, AUSTIN, or NEW YORK CITY, unless they have already provided one of these locations in the conversation. Once the location is known, take into account the user's preferences, including cuisine, budget, dietary restrictions, food allergies, vegan/vegetarian needs, gluten-free requirements, desired menu items, restaurant type, atmosphere, occasion, and other dining preferences, and use the restaurant knowledge base to find and recommend 3 restaurant best matching those preferences unless there are no matches in the knowledge base. Prioritize the user's explicit requirements, especially allergies and dietary restrictions, and briefly explain why each recommendation fits. For every restaurant recommended, provide its official restaurant website link when available. Use only information supported by the knowledge base and available restaurant website information; never invent restaurants, menu items, ingredients, prices, allergens, dietary accommodations, or website links, and clearly state when information is unknown or unconfirmed. For allergy-related questions, distinguish between confirmed allergens, allergen-free claims, possible cross-contact, and unknown information, and advise users to confirm serious allergies directly with the restaurant. Ask at most one follow-up question total before giving recommendations. If the user has already given you a cuisine, budget, dietary need, atmosphere, or any general preference, proceed to recommend restaurants from CONTEXT rather than asking further questions — give your best 2-3 matches and note if any unspecified detail (like budget) wasn't provided. When there is no perfect match, provide the closest options and explain what requirement may not be satisfied. Your goal is to act like a smart restaurant concierge: understand the user's location and preferences, filter and rank restaurants based on those preferences, and provide practical, personalized recommendations with official restaurant website links. Before answering, check every restaurant name, menu item, allergen note, and website URL against the CONTEXT block provided below. If a detail is not present in CONTEXT, do not include it — state it is not available in the knowledge base rather than guessing or using general knowledge. If CONTEXT does not contain enough restaurants matching the user's request,  say so explicitly and offer the closest available options from CONTEXT only — do not supplement with restaurants you know about from general knowledge. Only include a website URL if it appears verbatim in CONTEXT. If no URL is  listed for a restaurant, write "website not listed" — never construct or guess a URL or location.


RELEVANT KNOWLEDGE BASE ENTRIES FOR THIS QUERY:
{context}


Only use restaurants, menu items, and details that appear above. If the answer isn't in these entries, say so — do not guess or use outside knowledge."""



   messages = [{"role": "system", "content": system_prompt}]



   if history:
       messages.extend(history)



   messages.append({"role": "user", "content": message})



   response = client.chat_completion(
       messages,
       max_tokens=500,
       temperature=.7 #why: 1 was too creative, .5 too restrictive
   )



   return response.choices[0].message.content.strip()



theme = gr.themes.ThemeClass.from_hub("hmb/vaporwave")






#google maps embed prep, get location of restaurants:
def get_address_for_restaurant(restaurant_name, text_chunks):
   for chunk in text_chunks:
       parts = [p.strip() for p in chunk.split("|")]
       if len(parts) < 3:
           continue



       name_field = re.sub(r"^\[.*?\]\s*", "", parts[0]).strip()
       if name_field.lower() != restaurant_name.strip().lower():
           continue



       # CHAIN rows: parts[1] is the URL, parts[2] is the address.
       # SMALL BUSINESS rows: parts[1] is the address, parts[2] is the URL.
       # Detect by checking which field looks like a URL.
       if parts[1].startswith("http") or "." in parts[1].split()[0]:
           address_field = parts[2]
       else:
           address_field = parts[1]



       paren_match = re.search(r"\((.*?)\)", address_field)
       if paren_match:
           addresses = [a.strip() for a in paren_match.group(1).split(";")]
           return {"address": addresses[0], "all_addresses": addresses}



       return {"address": address_field, "all_addresses": [address_field]}



   return None



def sanitize_markdown(text):
   """
   The LLM sometimes emits unmatched single asterisks, which markdown
   interprets as opening italics that don't close until much later in
   the text — producing a huge unintended italic span. Combined with
   this theme's italic font (tight/invisible word spacing), that reads
   as squished, illegible text. This escapes any '*' that isn't part
   of a '**bold**' pair, so only intentional bold survives.
   """
   protected = text.replace("**", "\x00BOLD\x00")
   protected = protected.replace("*", "\\*")
   return protected.replace("\x00BOLD\x00", "**")






with gr.Blocks() as chatbot:



   gr.HTML("""
   <div style="text-align:center; width:100%;">
   <h1 style="font-size:48px; margin-bottom:0;">🍜 NomNom</h1>
   <p style="font-size:18px; color:gray; margin-top:0;">Your Food-Allergy Friendly Restaurant & Dish Finder</p>
   </div>
   """)



   dark_toggle = gr.Button("☀️ Toggle Mode 🌙", size="sm")
   dark_toggle.click(
       None, None, None,
       js=""" () => { document.body.classList.toggle('dark'); }""")



   with gr.Row() as location_row: #will be at the top of our bot
       location_label = gr.Markdown("**Choose your location to get started:**")
       loc_la = gr.Button("Los Angeles, CA")
       loc_nyc = gr.Button("New York City, NY")
       loc_austin = gr.Button("Austin, TX")



   filters_toggle = gr.Button("⚙️ Change Location", visible=False, size="sm")



   with gr.Group():
       chat = gr.Chatbot()
       msg = gr.Textbox(placeholder="Ask about restaurants...")



   with gr.Row():
       b1 = gr.Button(visible=False)
       b2 = gr.Button(visible=False)
       b3 = gr.Button(visible=False)



   selected_restaurant = gr.State("")



   def build_map_markdown(restaurant_name, address):
       query = f"{restaurant_name}, {address}".replace(" ", "+")
       map_url = (
           f"https://maps.googleapis.com/maps/api/staticmap"
           f"?center={query}&zoom=15&size=500x300&markers={query}"
           f"&key={GOOGLE_MAPS_API_KEY}"
       )
       return f"![{restaurant_name} location]({map_url})"






   def user_turn(message, history):
       reply = respond(message, history)
       reply = sanitize_markdown(reply)
       names = extract_known_restaurant_names(reply, cleaned_chunks)



       maps_markdown = []
       for name in names:
           location_info = get_address_for_restaurant(name, cleaned_chunks)
           if location_info:
               maps_markdown.append(build_map_markdown(name, location_info["address"]))
               if len(location_info["all_addresses"]) > 1:
                   other_locations = ", ".join(location_info["all_addresses"])
                   maps_markdown.append(
                       f"*{name} has multiple locations ({other_locations}). "
                       f"Map shown is for {location_info['address']}.*"
                   )



       content = reply
       if maps_markdown:
           content += "\n\n" + "\n\n".join(maps_markdown)



       history = history + [
           {"role": "user", "content": message},
           {"role": "assistant", "content": content},
       ]



       updates = []
       for i in range(3):
           if i < len(names):
               updates.append(gr.update(value=names[i], visible=True))
           else:
               updates.append(gr.update(visible=False))



       return (history, "", *updates,
               gr.update(visible=False), gr.update(visible=False),
               gr.update(visible=False), gr.update(visible=False),
               gr.update(visible=True))



   def pick_restaurant(name, history):
       message = f"Tell me more about {name}"
       reply = respond(message, history)
       reply = sanitize_markdown(reply)



       location_info = get_address_for_restaurant(name, cleaned_chunks)
       content = reply
       if location_info:
           content += "\n\n" + build_map_markdown(name, location_info["address"])
           if len(location_info["all_addresses"]) > 1:
               other_locations = ", ".join(location_info["all_addresses"])
               content += (
                   f"\n\n*{name} has multiple locations ({other_locations}). "
                   f"Map shown is for {location_info['address']}.*"
               )



       history = history + [
           {"role": "user", "content": message},
           {"role": "assistant", "content": content} #was: reply --> content is map
       ]
       # show feature buttons now that a restaurant is selected
       return history, "", gr.update(visible=False), gr.update(visible=False), gr.update(visible=False), gr.update(visible=True), name

   with gr.Row(visible=False) as feature_row:
       feat_allergens = gr.Button("🥗 Allergens & Dietary")
       feat_location = gr.Button("📍 Location & Hours")
       feat_menu = gr.Button("🍽️ Menu Highlights")
       feat_website = gr.Button("🔗 Website")



   b1.click(pick_restaurant, [b1, chat], [chat, msg, b1, b2, b3, feature_row, selected_restaurant])
   b2.click(pick_restaurant, [b2, chat], [chat, msg, b1, b2, b3, feature_row, selected_restaurant])
   b3.click(pick_restaurant, [b3, chat], [chat, msg, b1, b2, b3, feature_row, selected_restaurant])

   def ask_feature(feature_label, restaurant_name, history): #routed to restaurants
       message = f"What are the {feature_label} for {restaurant_name}?"
       reply = respond(message, history)
       reply = sanitize_markdown(reply)
       history = history + [
           {"role": "user", "content": message},
           {"role": "assistant", "content": reply}
       ]
       return history, ""

   feat_allergens.click(
       lambda name, hist: ask_feature("allergens and dietary accommodations", name, hist),
       [selected_restaurant, chat], [chat, msg]
   )
   feat_location.click(
       lambda name, hist: ask_feature("location and hours", name, hist),
       [selected_restaurant, chat], [chat, msg]
   )
   feat_menu.click(
       lambda name, hist: ask_feature("full menu", name, hist),
       [selected_restaurant, chat], [chat, msg]
   )
   feat_website.click(
       lambda name, hist: ask_feature("website", name, hist),
       [selected_restaurant, chat], [chat, msg]
   )



   with gr.Accordion("🔍 Filters", open=False) as filter_panel:
       allergy_filters = gr.CheckboxGroup(
           choices=["Gluten-free", "Dairy-free", "Nut-free", "Peanut-free", "Soy-free", "Wheat-free", "Sesame-free", "Shellfish-free", "Fish-free", "Egg-free", "Vegan", "Vegetarian"],
           label="Dietary needs"
           )
       budget_filter = gr.Radio(
           choices=["$", "$$", "$$$"],
           label="Budget"
       )
       apply_filters_btn = gr.Button("Apply Filters")



   def apply_filters(allergies, budget, history): #selectable
       parts = []
       if allergies:
           parts.append("with these dietary needs: " + ", ".join(allergies))
       if budget:
           parts.append(f"in the {budget} price range")
       if not parts:
           return history, ""
       message = "Find restaurants " + " and ".join(parts)
       reply = respond(message, history)
       reply = sanitize_markdown(reply)
       history = history + [
           {"role": "user", "content": message},
           {"role": "assistant", "content": reply}
       ]
       return history, ""

   apply_filters_btn.click(apply_filters, [allergy_filters, budget_filter, chat], [chat, msg])

   def show_filters():
       return (gr.update(visible=True), gr.update(visible=True),
               gr.update(visible=True), gr.update(visible=True),
               gr.update(visible=False))
   filters_toggle.click(show_filters, None, [location_label, loc_la, loc_nyc, loc_austin, filters_toggle])



   loc_la.click(user_turn, [loc_la, chat], [chat, msg, b1, b2, b3, location_label, loc_la, loc_nyc, loc_austin, filters_toggle])
   loc_nyc.click(user_turn, [loc_nyc, chat], [chat, msg, b1, b2, b3, location_label, loc_la, loc_nyc, loc_austin, filters_toggle])
   loc_austin.click(user_turn, [loc_austin, chat], [chat, msg, b1, b2, b3, location_label, loc_la, loc_nyc, loc_austin, filters_toggle])
   msg.submit(user_turn, [msg, chat], [chat, msg, b1, b2, b3, location_label, loc_la, loc_nyc, loc_austin, filters_toggle])



chatbot.launch(debug=True, theme=theme, share=True)

SecretNotFoundError: Secret HF_TOKEN does not exist.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')